# 📥 Download and Filter NASA GeneLab Omics Datasets

This notebook automates the retrieval and pre‑processing of omics datasets from the NASA GeneLab Open Science Data Repository (OSDR) using the `genelab_utils` package. It supports both incremental and full updates, applies pre‑filters to reduce file size, and writes a manifest of downloaded files.

Author: Peter W. Rose, UC San Diego (pwrose.ucsd@gmail.com)

In [18]:
import pandas as pd
import genelab_utils as gl

In [19]:
pd.set_option('display.max_rows', None)  # Shows all rows
pd.set_option('display.max_colwidth', None)  # Shows full content of each cell

In [20]:
MANIFEST_PATH = "../data/manifest.csv" # file to save dataset info

## Incremental vs Full Update
By default, this notebook runs an incremental update. It downloads and preprocesses any new datasets specified in the "technology_types" list below.

If any datasets have been updated, set the "reset" variable to "True" to run a complete update.

The downloaded datasets are saved in the "datasets" directory.

In [21]:
RESET = False # run incremental update
# RESET = True # run a complete update to refresh datasets

In [22]:
info = gl.get_info()
info.head()

,identifier,assay_name,id.sample name,measurement,technology,material,study.characteristics.material type.term accession number,study.characteristics.organism.term accession number,file.category,file.subcategory,taxonomy
0,OSD-1,OSD-1_transcription-profiling_dna-microarray_Affymetrix,Dmel_OR_wo_FLT_infdw-Bbas_Rep1,transcription profiling,DNA microarray,Whole Organism,http://purl.obolibrary.org/obo/NCIT_C13413,http://purl.bioontology.org/ontology/NCBITAXON/7227,GeneLab Processed Microarray Data Files,Differential gene expression analysis,7227
1,OSD-1,OSD-1_transcription-profiling_dna-microarray_Affymetrix,Dmel_OR_wo_GC_infdw-Bbas_Rep3,transcription profiling,DNA microarray,Whole Organism,http://purl.obolibrary.org/obo/NCIT_C13413,http://purl.bioontology.org/ontology/NCBITAXON/7227,GeneLab Processed Microarray Data Files,Normalized Data,7227
2,OSD-1,OSD-1_transcription-profiling_dna-microarray_Affymetrix,Dmel_OR_wo_GC_infdw-Bbas_Rep3,transcription profiling,DNA microarray,Whole Organism,http://purl.obolibrary.org/obo/NCIT_C13413,http://purl.bioontology.org/ontology/NCBITAXON/7227,GeneLab Processed Microarray Data Files,Differential gene expression analysis,7227
3,OSD-1,OSD-1_transcription-profiling_dna-microarray_Affymetrix,Dmel_OR_wo_GC_infdw-Bbas_Rep3,transcription profiling,DNA microarray,Whole Organism,http://purl.obolibrary.org/obo/NCIT_C13413,http://purl.bioontology.org/ontology/NCBITAXON/7227,Microarray Data Files,Normalized data,7227
4,OSD-1,OSD-1_transcription-profiling_dna-microarray_Affymetrix,Dmel_OR_wo_GC_infdw-Bbas_Rep3,transcription profiling,DNA microarray,Whole Organism,http://purl.obolibrary.org/obo/NCIT_C13413,http://purl.bioontology.org/ontology/NCBITAXON/7227,Microarray Data Files,Raw array data,7227


## Get a List of GeneLab processed Datasets

In [23]:
# df = gl.get_info()
# df = df.query("identifier == 'OSD-679'")
# df

In [24]:
dataset_info = gl.get_processed_datasets()

In [25]:
dataset_info.head()

,identifier,assay_name,id.sample count,measurement,technology,material,study.characteristics.material type.term accession number,study.characteristics.organism.term accession number,file.GL-processed,file.non-GL-processed,taxonomy
189,OSD-1,OSD-1_transcription-profiling_dna-microarray_Affymetrix,18,transcription profiling,DNA microarray,Whole Organism,http://purl.obolibrary.org/obo/NCIT_C13413,http://purl.bioontology.org/ontology/NCBITAXON/7227,GeneLab Processed Microarray Data Files,False,7227
279,OSD-100,OSD-100_transcription-profiling_rna-sequencing-(rna-seq),12,transcription profiling,RNA Sequencing (RNA-Seq),left eye,http://purl.obolibrary.org/obo/UBERON_0004548,http://purl.bioontology.org/ontology/NCBITAXON/10090,GeneLab Processed RNA-Seq Files,False,10090
286,OSD-101,OSD-101_transcription-profiling_rna-sequencing-(rna-seq)_Illumina,12,transcription profiling,RNA Sequencing (RNA-Seq),Left gastrocnemius,http://purl.org/sig/ont/fma/fma22557,http://purl.bioontology.org/ontology/NCBITAXON/10090,GeneLab Processed RNA-Seq Files,False,10090
281,OSD-102,OSD-102_transcription-profiling_rna-sequencing-(rna-seq)_Illumina HiSeq 4000,12,transcription profiling,RNA Sequencing (RNA-Seq),Left kidney,http://purl.org/sig/ont/fma/fma7205,http://purl.bioontology.org/ontology/NCBITAXON/10090,GeneLab Processed RNA-Seq Files,False,10090
265,OSD-103,OSD-103_dna-methylation-profiling_whole-genome-bisulfite-sequencing,12,DNA methylation profiling,Whole Genome Bisulfite Sequencing,Quadriceps-left,NaN,http://purl.bioontology.org/ontology/NCBITAXON/10090,GeneLab Processed Whole Genome Bisulfite-Seq Files,False,10090


## Filter by Technology Type

In [26]:
technology_types = ["Magnetic Resonance Imaging",]

dataset_info = gl.filter_by_technology_type(info, technology_types)

## Filter by Organism

In [27]:
print(f"Available organisms: {dataset_info['taxonomy'].unique()}")

Available organisms: ['10116']


In [28]:
taxids = {"9606": "Homo sapiens",
          # -- Rodens -- 
          "10090": "Mus musculus",
          "10116": "Rattus norvegicus",
          # -- Fish --
          # "7955": "Danio rerio",
          "8090": "Oryzias latipes",
          # -- Nematoda --
          # "6239": "Caenorhabditis elegans",
          # -- Insecta --
          # "7227": "Drosophila melanogaster",
          # "63436": "Leptopilina heterotoma",
          # "63433": "Leptopilina boulardi",
          # -- Bacteria --
          "562": "Escherichia coli",
          "287": "Pseudomonas aeruginosa",
          "1423": "Bacillus subtilis",
          "1781": "Mycobacterium marinum",
          "148447": "Paraburkholderia phymatum",
          # -- Fungi --
          # "4932": "Saccharomyces cerevisiae",
          # -- Plants --
          # "3711": "Brassica rapa",
          # "15368": Brachypodium distachyon",
          # "3702": "Arabidopsis thaliana",
         }
          
dataset_info = gl.filter_by_organism(info, taxids)

In [29]:
print(f"Filtered organisms: {dataset_info['taxonomy'].unique()}")

Filtered organisms: ['9606' '10090' '10116' '287' '1423' '562' '8090' '1781' '148447']


In [30]:
dataset_info = dataset_info[["identifier", "technology", "measurement", "assay_name", "taxonomy", "organism", "material"]].copy()
dataset_info.drop_duplicates(inplace=True)
dataset_info.head()

,identifier,technology,measurement,assay_name,taxonomy,organism,material
198,OSD-2,DNA microarray,transcription profiling,OSD-2_transcription-profiling_dna-microarray_agilent,9606,Homo sapiens,Cells
366,OSD-4,DNA microarray,transcription profiling,OSD-4_transcription-profiling_dna-microarray_affymetrix,10090,Mus musculus,thymus
438,OSD-5,DNA microarray,transcription profiling,OSD-5_transcription-profiling_dna-microarray_affymetrix,9606,Homo sapiens,primary cell
444,OSD-6,DNA microarray,transcription profiling,OSD-6_transcription-profiling_dna-microarray_affymetrix,10116,Rattus norvegicus,Cells
716,OSD-9,DNA microarray,transcription profiling,OSD-9_transcription-profiling_dna-microarray_Compugen,9606,Homo sapiens,NaN


## Select Datasets to Download
The map below specifies the technology type and a substring used to identify processed files. Processed files must contain this substring.

In [31]:
# file_types = {"DNA microarray": "differential_expression",
#               "RNA Sequencing (RNA-Seq)": "differential_expression",
#               "Whole Genome Bisulfite Sequencing": "differential_methylation_tiles",
#               "Reduced-Representation Bisulfite Sequencing": "differential_methylation_tiles",}
file_types = {"Magnetic Resonance Imaging": "TRANSFORMED",}

#### Define pre-filters to reduce the file the essential data

In [32]:
def differential_expression_filter(df, threshold=0.05):
    filtered_df = df[df['ENTREZID'].notna() & (df['ENTREZID'].astype(str) != '')]
    # Keep only required columns
    filtered_df = filtered_df.filter(regex=r"^(ENTREZID|GENENAME|Log2fc_|Adj\.p\.value_)")
    adj_pval_cols = [col for col in filtered_df.columns if col.startswith("Adj.p.value_")]
    filtered_df = filtered_df[filtered_df[adj_pval_cols].le(threshold).any(axis=1)]
    # Explode rows with multiple genes
    if "ENTREZID" in filtered_df.columns:
        filtered_df["ENTREZID"] = filtered_df["ENTREZID"].astype(str)
        filtered_df["ENTREZID"] = filtered_df["ENTREZID"].apply(lambda x:x.split('|'))
        filtered_df = filtered_df.explode('ENTREZID')
        filtered_df["ENTREZID"] = filtered_df["ENTREZID"].str.strip()
    return filtered_df

In [33]:
def differential_methylation_filter(df, threshold=0.05):
    filtered_df = df[df['ENTREZID'].notna() & (df['ENTREZID'].astype(str) != '')]
    # Keep only required columns
    filtered_df = filtered_df.filter(regex=r"^(ENTREZID|GENENAME|chr|start|end|dist.to.feature|prom|exon|intron|meth.diff_|qvalue_)")
    qval_cols = [col for col in filtered_df.columns if col.startswith("qvalue_")]
    filtered_df = filtered_df[filtered_df[qval_cols].le(threshold).any(axis=1)]
     # Explode rows with multiple genes
    if "ENTREZID" in filtered_df.columns:
        filtered_df["ENTREZID"] = filtered_df["ENTREZID"].astype(str)
        filtered_df["ENTREZID"] = filtered_df["ENTREZID"].apply(lambda x:x.split('|'))
        filtered_df = filtered_df.explode('ENTREZID')
        filtered_df["ENTREZID"] = filtered_df["ENTREZID"].str.strip()
    return filtered_df

In [34]:
def tomography_filter(df):
    return df

In [35]:
filters = {"differential_expression": differential_expression_filter,
           "differential_methylation_tiles": differential_methylation_filter,
           "TRANSFORMED": tomography_filter}

In [36]:
manifest = gl.download_data_files(dataset_info, file_types, filters, reset=RESET)
manifest.to_csv(MANIFEST_PATH, index=False)

Downloading: LSDS-81_tonometry_Fuller_TonometryIOP_TRANSFORMED.csv
Downloading: LSDS-81_Ophthalmologic Diagnostic Technique_Fuller_OCT_TRANSFORMED.csv
Downloading: LSDS-81_Tomography_Fuller_MRI_Eye_TRANSFORMED.csv
File already exist: LSDS-81_tonometry_Fuller_TonometryIOP_TRANSFORMED.csv
File already exist: LSDS-81_Ophthalmologic Diagnostic Technique_Fuller_OCT_TRANSFORMED.csv
File already exist: LSDS-81_Tomography_Fuller_MRI_Eye_TRANSFORMED.csv
Downloading: LSDS-82_Tomography_Fuller_MRI_OpticNerve_TRANSFORMED.csv
File already exist: LSDS-82_Tomography_Fuller_MRI_OpticNerve_TRANSFORMED.csv


In [37]:
manifest.head()

,identifier,technology,measurement,assay_name,taxonomy,organism,material,filename,url
246643,OSD-679,Magnetic Resonance Imaging,Tomography,OSD-679_tomography_magnetic-resonance-imaging_bruker biospec 7t,10116,Rattus norvegicus,Right Eye,LSDS-81_tonometry_Fuller_TonometryIOP_TRANSFORMED.csv,https://osdr.nasa.gov/geode-py/ws/studies/OSD-679/download?source=datamanager&file=LSDS-81_tonometry_Fuller_TonometryIOP_TRANSFORMED.csv
246643,OSD-679,Magnetic Resonance Imaging,Tomography,OSD-679_tomography_magnetic-resonance-imaging_bruker biospec 7t,10116,Rattus norvegicus,Right Eye,LSDS-81_Ophthalmologic Diagnostic Technique_Fuller_OCT_TRANSFORMED.csv,https://osdr.nasa.gov/geode-py/ws/studies/OSD-679/download?source=datamanager&file=LSDS-81_Ophthalmologic+Diagnostic+Technique_Fuller_OCT_TRANSFORMED.csv
246643,OSD-679,Magnetic Resonance Imaging,Tomography,OSD-679_tomography_magnetic-resonance-imaging_bruker biospec 7t,10116,Rattus norvegicus,Right Eye,LSDS-81_Tomography_Fuller_MRI_Eye_TRANSFORMED.csv,https://osdr.nasa.gov/geode-py/ws/studies/OSD-679/download?source=datamanager&file=LSDS-81_Tomography_Fuller_MRI_Eye_TRANSFORMED.csv
246646,OSD-679,Magnetic Resonance Imaging,Tomography,OSD-679_tomography_magnetic-resonance-imaging_bruker biospec 7t,10116,Rattus norvegicus,Left Eye,LSDS-81_tonometry_Fuller_TonometryIOP_TRANSFORMED.csv,https://osdr.nasa.gov/geode-py/ws/studies/OSD-679/download?source=datamanager&file=LSDS-81_tonometry_Fuller_TonometryIOP_TRANSFORMED.csv
246646,OSD-679,Magnetic Resonance Imaging,Tomography,OSD-679_tomography_magnetic-resonance-imaging_bruker biospec 7t,10116,Rattus norvegicus,Left Eye,LSDS-81_Ophthalmologic Diagnostic Technique_Fuller_OCT_TRANSFORMED.csv,https://osdr.nasa.gov/geode-py/ws/studies/OSD-679/download?source=datamanager&file=LSDS-81_Ophthalmologic+Diagnostic+Technique_Fuller_OCT_TRANSFORMED.csv
